# Human PPI: embedding geometry vs experimental essentiality

**Goal:** Check whether **hyperbolic radius** and simple **chart radii** differ between genes labeled **essential** vs **non-essential** in a consolidated human screen dataset (SNAP / OGEE-style table), for **D=2** and **D=3** D-Mercator runs under ``d-mercator-run/d2/`` and ``d3/``.

**Inputs:** ``resources/G-HumanEssential.tsv.gz`` + ``resources/Homo_sapiens.gene_info.gz`` (see ``resources/SOURCES.md``). Run ``python fetch_resources.py`` from this folder if files are missing.

**Working directory:** ``notebooks/validation/`` (or adjust ``REPO`` in the first code cell).


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

def _find_repo() -> Path:
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "d-mercator-run").is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise RuntimeError("Set cwd to repo or notebooks/validation (need d-mercator-run/). cwd=" + str(Path.cwd()))


REPO = _find_repo()
sys.path.insert(0, str(REPO / "notebooks" / "dmercator"))
sys.path.insert(0, str(REPO / "notebooks" / "dmercator3d"))
sys.path.insert(0, str(REPO / "notebooks" / "validation"))

from embedding_essentiality import attach_essentiality, disk_radius_from_ortho_xy, load_snap_essentiality

VAL = REPO / "notebooks" / "validation"
RES = VAL / "resources"
assert (RES / "G-HumanEssential.tsv.gz").is_file(), "Run: python fetch_resources.py"
assert (RES / "Homo_sapiens.gene_info.gz").is_file(), "Run: python fetch_resources.py"

ess = load_snap_essentiality()
print("SNAP rows (symbol-mapped):", len(ess))
print(ess["essential"].value_counts())


## Native **D = 2** (`d-mercator-run/d2/`)

**Geometry:** `Inf.Hyp.Rad` (hyperbolic radial in the model) and **orthographic disk radius** $\sqrt{x^2+y^2}$ after `dmercator_io.ortho_xy_disk` (same chart family as the 2D disk export).

**Statistics:** Mann–Whitney on `Inf.Hyp.Rad`, point-biserial correlation, optional logistic regression controlling for **graph degree**.


In [ ]:
import dmercator_io as dm

paths = dm.paths_for_run("d2")
meta, df = dm.parse_inf_coord(paths["inf_coord"])
G = dm.load_edges_graph(paths["edge"])
deg_map = {str(a): int(b) for a, b in G.degree()}
df = df.assign(degree=df["Vertex"].astype(str).map(deg_map))
df["degree"] = df["degree"].fillna(0).astype(int)
x, y = dm.ortho_xy_disk(df)
df["r_ortho_disk"] = disk_radius_from_ortho_xy(df, x, y)

m = attach_essentiality(df, ess=ess)
labeled = m[m["essential"].notna()].copy()
labeled["essential"] = labeled["essential"].astype(int)
print("D=2 | vertices", len(m), "| labeled", len(labeled), "| essential", int(labeled["essential"].sum()))

fig, axes = plt.subplots(1, 2, figsize=(9, 4), dpi=150)
for ax, col, title in zip(
    axes,
    ["Inf.Hyp.Rad", "r_ortho_disk"],
    ["Inf.Hyp.Rad", "Orthographic disk radius sqrt(x^2+y^2)"],
):
    v0 = labeled.loc[labeled["essential"] == 0, col].to_numpy()
    v1 = labeled.loc[labeled["essential"] == 1, col].to_numpy()
    ax.violinplot([v0, v1], positions=[0, 1], showmeans=True)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["non-essential", "essential"])
    ax.set_ylabel(col)
    ax.set_title(title)
fig.suptitle("Native D=2 run (d2)")
fig.tight_layout()
plt.show()

g0 = labeled.loc[labeled["essential"] == 0, "Inf.Hyp.Rad"]
g1 = labeled.loc[labeled["essential"] == 1, "Inf.Hyp.Rad"]
u_stat, u_p = stats.mannwhitneyu(g1, g0, alternative="two-sided")
print("Mann–Whitney Inf.Hyp.Rad (essential vs non): U=", u_stat, "p=", u_p)

r_pb, r_p = stats.pointbiserialr(labeled["essential"], labeled["Inf.Hyp.Rad"])
print("Point-biserial essential x Inf.Hyp.Rad: r=", r_pb, "p=", r_p)

c_pb, c_p = stats.pointbiserialr(labeled["essential"], labeled["r_ortho_disk"])
print("Point-biserial essential x disk radius: r=", c_pb, "p=", c_p)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

Xf = labeled[["Inf.Hyp.Rad", "degree"]].to_numpy(dtype=float)
y = labeled["essential"].to_numpy()
Xs = StandardScaler().fit_transform(Xf)
clf = LogisticRegression(max_iter=300)
clf.fit(Xs, y)
print("Logistic coef (scaled): Inf.Hyp.Rad, degree:", clf.coef_[0])


## Native **D = 3** (`d-mercator-run/d3/`)

**Geometry:** `Inf.Hyp.Rad` and **stereographic ball radius** $\|\mathbf{y}\|$ in $\mathbb{R}^3$ from `ball_projection.stereographic_s3_to_r3` on unit directions (north pole chart).

**Same tests** as D=2 for comparability.


In [ ]:
import dmercator3d_io as dm
from ball_projection import stereographic_s3_to_r3

paths = dm.paths_for_run("d3")
meta, df = dm.parse_inf_coord(paths["inf_coord"])
G = dm.load_edges_graph(paths["edge"])
deg_map = {str(a): int(b) for a, b in G.degree()}
df = df.assign(degree=df["Vertex"].astype(str).map(deg_map))
df["degree"] = df["degree"].fillna(0).astype(int)
U = dm.normalize_direction_nd(df)
x1, x2, x3, x4 = U[:, 0], U[:, 1], U[:, 2], U[:, 3]
Xb, Yb, Zb = stereographic_s3_to_r3(x1, x2, x3, x4, pole="north")
df["r_stereo_ball"] = np.sqrt(Xb * Xb + Yb * Yb + Zb * Zb)

m = attach_essentiality(df, ess=ess)
labeled = m[m["essential"].notna()].copy()
labeled["essential"] = labeled["essential"].astype(int)
print("D=3 | vertices", len(m), "| labeled", len(labeled), "| essential", int(labeled["essential"].sum()))

fig, axes = plt.subplots(1, 2, figsize=(9, 4), dpi=150)
for ax, col, title in zip(
    axes,
    ["Inf.Hyp.Rad", "r_stereo_ball"],
    ["Inf.Hyp.Rad", "Stereographic R3 ball radius ||y||"],
):
    v0 = labeled.loc[labeled["essential"] == 0, col].to_numpy()
    v1 = labeled.loc[labeled["essential"] == 1, col].to_numpy()
    ax.violinplot([v0, v1], positions=[0, 1], showmeans=True)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["non-essential", "essential"])
    ax.set_ylabel(col)
    ax.set_title(title)
fig.suptitle("Native D=3 run (d3)")
fig.tight_layout()
plt.show()

g0 = labeled.loc[labeled["essential"] == 0, "Inf.Hyp.Rad"]
g1 = labeled.loc[labeled["essential"] == 1, "Inf.Hyp.Rad"]
u_stat, u_p = stats.mannwhitneyu(g1, g0, alternative="two-sided")
print("Mann–Whitney Inf.Hyp.Rad (essential vs non): U=", u_stat, "p=", u_p)

r_pb, r_p = stats.pointbiserialr(labeled["essential"], labeled["Inf.Hyp.Rad"])
print("Point-biserial essential x Inf.Hyp.Rad: r=", r_pb, "p=", r_p)

c_pb, c_p = stats.pointbiserialr(labeled["essential"], labeled["r_stereo_ball"])
print("Point-biserial essential x stereo ball radius: r=", c_pb, "p=", c_p)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

Xf = labeled[["Inf.Hyp.Rad", "degree"]].to_numpy(dtype=float)
y = labeled["essential"].to_numpy()
Xs = StandardScaler().fit_transform(Xf)
clf = LogisticRegression(max_iter=300)
clf.fit(Xs, y)
print("Logistic coef (scaled): Inf.Hyp.Rad, degree:", clf.coef_[0])


## Native **D=2 vs D=3** on matched vertices

**Setup:** Same `edges_GC` graph; parse **`d2`** and **`d3`** `inf_coord`, **inner-join** on `Vertex`. Metrics: **`Inf.Hyp.Rad`** each run, **2D orthographic disk radius** √(x²+y²), **3D stereographic ball radius** ‖y‖, and **degree** (from the shared edgelist).

**Figures (this cell):** (1) scatter **`hyp_d2` vs `hyp_d3`** with **identity** line and **OLS** fit; (2) **Bland–Altman** plot for hyperbolic radii; (3) **violin** of **Δhyp = hyp_d3 − hyp_d2** by essentiality; (4) **correlation heatmap** of numeric columns on labeled genes; (5) **ROC curves** for single-feature scores (sign flipped so higher ⇒ more essential when correlation is negative); (6) **bar chart of ROC AUC**.

**Requires:** earlier cells executed so `REPO`, `sys.path`, and **`ess`** exist.


In [ ]:
# Matched D=2 vs D=3 comparison (run after setup cell with `ess`)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import dmercator_io as dm2
import dmercator3d_io as dm3
from ball_projection import stereographic_s3_to_r3
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import auc, roc_curve
from sklearn.preprocessing import StandardScaler

paths2 = dm2.paths_for_run("d2")
paths3 = dm3.paths_for_run("d3")
_, df2 = dm2.parse_inf_coord(paths2["inf_coord"])
_, df3 = dm3.parse_inf_coord(paths3["inf_coord"])
G = dm2.load_edges_graph(paths2["edge"])
deg_map = {str(a): int(b) for a, b in G.degree()}


def prep_d2(df):
    x, y = dm2.ortho_xy_disk(df)
    r_disk = np.sqrt(np.asarray(x) ** 2 + np.asarray(y) ** 2)
    d = df.assign(degree=df["Vertex"].astype(str).map(deg_map))
    d["degree"] = d["degree"].fillna(0).astype(int)
    return pd.DataFrame(
        {
            "Vertex": df["Vertex"],
            "hyp_d2": df["Inf.Hyp.Rad"].astype(float),
            "r_disk_d2": r_disk,
            "degree": d["degree"],
        }
    )


def prep_d3(df):
    U = dm3.normalize_direction_nd(df)
    x1, x2, x3, x4 = U[:, 0], U[:, 1], U[:, 2], U[:, 3]
    Xb, Yb, Zb = stereographic_s3_to_r3(x1, x2, x3, x4, pole="north")
    r_ball = np.sqrt(Xb * Xb + Yb * Yb + Zb * Zb)
    return pd.DataFrame(
        {
            "Vertex": df["Vertex"],
            "hyp_d3": df["Inf.Hyp.Rad"].astype(float),
            "r_ball_d3": r_ball,
        }
    )


A = prep_d2(df2)
B = prep_d3(df3)
M = A[["Vertex", "hyp_d2", "r_disk_d2", "degree"]].merge(
    B[["Vertex", "hyp_d3", "r_ball_d3"]], on="Vertex", how="inner"
)

E = ess.assign(_k=ess["gene_symbol"].astype(str).str.strip().str.upper()).drop_duplicates(subset=["_k"])[["_k", "essential"]]
comb = M.assign(_k=M["Vertex"].astype(str).str.strip().str.upper()).merge(E, on="_k", how="inner").drop(columns=["_k"])
lab = comb[comb["essential"].notna()].copy()
lab["essential"] = lab["essential"].astype(int)
print("matched vertices (inner join):", len(M))
print("with SNAP label:", len(lab))
print("Spearman hyp_d2 vs hyp_d3 (all matched):", *stats.spearmanr(M["hyp_d2"], M["hyp_d3"]))

# --- 1) Scatter hyp_d2 vs hyp_d3 + identity + OLS ---
fig1, ax = plt.subplots(figsize=(6.2, 6), dpi=150)
s0 = lab["essential"] == 0
s1 = lab["essential"] == 1
ax.scatter(lab.loc[s0, "hyp_d2"], lab.loc[s0, "hyp_d3"], s=8, alpha=0.25, c="0.45", label="non-essential")
ax.scatter(lab.loc[s1, "hyp_d2"], lab.loc[s1, "hyp_d3"], s=10, alpha=0.55, c="crimson", label="essential")
lo = float(min(lab["hyp_d2"].min(), lab["hyp_d3"].min()))
hi = float(max(lab["hyp_d2"].max(), lab["hyp_d3"].max()))
ax.plot([lo, hi], [lo, hi], "k--", lw=1.2, label="identity")
slope, intercept, r_val, p_val, _ = stats.linregress(lab["hyp_d2"], lab["hyp_d3"])
xs = np.linspace(lo, hi, 80)
ax.plot(xs, slope * xs + intercept, color="darkorange", lw=2, label=f"OLS (r={r_val:.3f}, p={p_val:.1e})")
ax.set_xlabel("Inf.Hyp.Rad (D=2)")
ax.set_ylabel("Inf.Hyp.Rad (D=3)")
ax.set_title("Hyperbolic radius: D=2 vs D=3 (labeled genes)")
ax.legend(loc="upper left", fontsize=8)
ax.set_aspect("equal", adjustable="box")
fig1.tight_layout()
plt.show()

# --- 2) Bland–Altman (hyp) ---
mean_h = (lab["hyp_d2"] + lab["hyp_d3"]) / 2.0
diff_h = lab["hyp_d3"] - lab["hyp_d2"]
md = float(np.median(diff_h))
mad = 1.4826 * float(np.median(np.abs(diff_h - md)))
fig2, ax = plt.subplots(figsize=(6.5, 4), dpi=150)
ax.scatter(mean_h[s0], diff_h[s0], s=8, alpha=0.25, c="C0", label="non-essential")
ax.scatter(mean_h[s1], diff_h[s1], s=10, alpha=0.45, c="C3", label="essential")
ax.axhline(md, color="k", lw=1)
ax.axhline(md + 1.96 * mad, color="k", ls="--", lw=0.9)
ax.axhline(md - 1.96 * mad, color="k", ls="--", lw=0.9)
ax.set_xlabel("mean(hyp_d2, hyp_d3)")
ax.set_ylabel("hyp_d3 − hyp_d2")
ax.set_title("Bland–Altman; robust median ± 1.96·MAD")
ax.legend(loc="upper right", fontsize=8)
fig2.tight_layout()
plt.show()

# --- 3) Violin Δhyp by essential ---
fig3, ax = plt.subplots(figsize=(5, 4), dpi=150)
parts = [
    lab.loc[s0, "hyp_d3"].to_numpy() - lab.loc[s0, "hyp_d2"].to_numpy(),
    lab.loc[s1, "hyp_d3"].to_numpy() - lab.loc[s1, "hyp_d2"].to_numpy(),
]
ax.violinplot(parts, positions=[0, 1], showmeans=True, showmedians=True)
ax.set_xticks([0, 1])
ax.set_xticklabels(["non-essential", "essential"])
ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("Δ Inf.Hyp.Rad (D3 − D2)")
ax.set_title("Embedding shift by essentiality")
fig3.tight_layout()
plt.show()

# --- 4) Correlation heatmap (labeled subset) ---
cols = ["hyp_d2", "hyp_d3", "r_disk_d2", "r_ball_d3", "degree"]
C = np.corrcoef(lab[cols].to_numpy().T)
fig4, ax = plt.subplots(figsize=(5.5, 4.8), dpi=150)
im = ax.imshow(C, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols)))
ax.set_yticks(range(len(cols)))
short = ["hyp2", "hyp3", "disk2", "ball3", "deg"]
ax.set_xticklabels(short, rotation=35, ha="right")
ax.set_yticklabels(short)
fig4.colorbar(im, ax=ax, fraction=0.046)
ax.set_title("Pearson correlation (labeled genes)")
fig4.tight_layout()
plt.show()

# --- 5–6) ROC + AUC bars ---
y = lab["essential"].to_numpy()
scores = {
    "−hyp_d2": -lab["hyp_d2"].to_numpy(),
    "−hyp_d3": -lab["hyp_d3"].to_numpy(),
    "−r_disk_d2": -lab["r_disk_d2"].to_numpy(),
    "−r_ball_d3": -lab["r_ball_d3"].to_numpy(),
    "degree": lab["degree"].to_numpy(),
}
fig5, ax = plt.subplots(figsize=(6, 5), dpi=150)
aucs = {}
for name, s in scores.items():
    fpr, tpr, _ = roc_curve(y, s)
    aucs[name] = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=1.5, label=f"{name} AUC={aucs[name]:.3f}")
ax.plot([0, 1], [0, 1], "k--", lw=0.8)
ax.set_xlabel("FPR")
ax.set_ylabel("TPR")
ax.set_title("ROC — single-feature (higher score ⇒ more essential)")
ax.legend(loc="lower right", fontsize=7)
fig5.tight_layout()
plt.show()

fig6, ax = plt.subplots(figsize=(6.2, 3.2), dpi=150)
names = list(aucs.keys())
vals = [aucs[n] for n in names]
colors = plt.cm.tab10(np.linspace(0, 1, len(names)))
ax.barh(names, vals, color=colors)
ax.axvline(0.5, color="k", ls=":", lw=0.9)
ax.set_xlim(0.4, max(0.55, max(vals) + 0.03))
ax.set_xlabel("ROC AUC")
ax.set_title("Single-feature AUC (same labeled set)")
for i, v in enumerate(vals):
    ax.text(v + 0.002, i, f"{v:.3f}", va="center", fontsize=8)
fig6.tight_layout()
plt.show()

Xb = np.column_stack(
    [
        StandardScaler().fit_transform(lab[["hyp_d2", "hyp_d3"]].to_numpy()),
        StandardScaler().fit_transform(lab[["degree"]].to_numpy()),
    ]
)
clf_b = LogisticRegression(max_iter=400)
clf_b.fit(Xb, y)
print("Logistic coef [hyp_d2, hyp_d3, degree] (std blocks):", clf_b.coef_[0])
